## Preprocessing PDF file to image

In [1]:
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
from surya.layout import LayoutPredictor
from PIL import ImageDraw
import pandas as pd


def draw_bounding_boxes(image, predictions, is_layout=False):
    draw = ImageDraw.Draw(image)
    # Rounded rectangle parameters
    border_radius = 0  # Corner radius
    outline_color = "red"  # Box color
    outline_width = 1  # Box thickness
    print(predictions)

    if is_layout:
        predictions = predictions[0].bboxes
    else:
        predictions = predictions[0].text_lines

    paragraph = ""

    # Loop through each set of coordinates
    for coords in predictions:
        # Extract x and y values
        x_values = [x for x, y in coords.polygon]
        y_values = [y for x, y in coords.polygon]

        # Calculate bounding box
        min_x = min(x_values)
        max_x = max(x_values)
        min_y = min(y_values)
        max_y = max(y_values)

        # Draw the rounded rectangle
        draw.rounded_rectangle(
            [(min_x, min_y), (max_x, max_y)],
            radius=border_radius,
            outline=outline_color,
            width=outline_width,
        )
        if is_layout is False:
            paragraph += coords.text + "\n"
    return image, paragraph


def rounding_box(image, is_layout=False):
    image_cp = image.copy()
    if is_layout:
        layout_predictor = LayoutPredictor()
        detection_predictor = DetectionPredictor()
        predictions = layout_predictor([image_cp])
    else:
        langs = [
            "en"
        ]  # Replace with your languages or pass None (recommended to use None)
        recognition_predictor = RecognitionPredictor()
        detection_predictor = DetectionPredictor()
        predictions = recognition_predictor(
            [image_cp], det_predictor=detection_predictor
        )

    image_output, paragraph = draw_bounding_boxes(image_cp, predictions, is_layout)
    return image_output, paragraph

/home/duckq1u/miniconda3/envs/OCR/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Get a topic from paragraph

In [2]:
"""title_list form:
{
    paragraph_title: "ten van ban",
    value: {
        1: {
            start_text: "so dang ky",
            end_text: "empty",
        },
        2: {
            start_text: "1.1 ben nhan bao dam",
            end_text: "1.2 ben bao dam",
        },
        3: {
            start_text: "1.2 ben bao dam",
            end_text: "1.3 ma pin",
        }
    }
}
"""

import re

json_test = [
    {
        "paragraph_title": "VĂN BẢN CHỨNG NHẬN ĐĂNG KÝ BIỆN PHÁP BẢO ĐẢM, THÔNG BÁO XỬ LÝ TÀI SẢN BẢO ĐẢM",
        "case": {
            "case_1": {
                1: {"start_text": "số đăng ký", "end_text": "empty"},
                2: {
                    "start_text": "1.1. Bên nhận bảo đảm",
                    "end_text": "1.2. Bên bảo đảm",
                },
                3: {"start_text": "1.2. Bên bảo đảm", "end_text": "1.3. Mã PIN:"},
            },
            "case_2": {
                1: {"start_text": "số đăng ký", "end_text": "empty"},
                2: {
                    "start_text": "1.1. Bên nhận thế chấp",
                    "end_text": "1.2. Bên thế chấp",
                },
                3: {"start_text": "1.2. Bên thế chấp", "end_text": "1.3. Mã PIN:"},
            }
        },
    }
]


def norm_text(text):
    # remove . and :
    text = re.sub(r"[.:]", "", text)
    return text.strip().lower()



def get_text_from_paragraph(paragraph: list, text_value: dict):
    paragraph_title = ""
    index_title = 0

    value = []
    temp_cache = ""
    is_get_value = False
    for index_line, line in enumerate(paragraph):
        # Detect paragraph title
        if paragraph_title == "":
            for index_key, title in enumerate(text_value):
                if norm_text(line) in norm_text(
                    title["paragraph_title"]
                ) or f"{norm_text(paragraph[index_line])} {norm_text(paragraph[index_line+1])}" == norm_text(
                    title["paragraph_title"]
                ):
                    paragraph_title = title["paragraph_title"]
                    index_title = index_key
                    break
                
    for case_index, case in enumerate(text_value[index_title]["case"]):
        if len(value) > 0: break
        # NOTE: If paragraph title is found, start extracting values
        for key in text_value[index_title]["case"][case]:
            title = text_value[index_title]["case"][case][key]
            for index_line, line in enumerate(paragraph):
                
                # NOTE: if is_get_value is True, we are in the process of extracting a value
                if is_get_value:
                    if norm_text(title["end_text"]) in norm_text(line):
                        if norm_text(title["end_text"]) != norm_text(line):
                            temp_cache += line.split(title["end_text"])[0].strip() + "\n"
                            value.append(temp_cache)
                            is_get_value = False
                            temp_cache = ""
                            continue
                        else:
                            value.append(temp_cache)
                            is_get_value = False
                            temp_cache = ""
                            continue
                    temp_cache += line + "\n"
                    continue
                elif norm_text(title["start_text"]) in norm_text(line):
                    if norm_text(title["start_text"]) != norm_text(line):
                        temp_cache += (
                            title["start_text"]
                            + " "
                            + line.split(title["start_text"])[-1].strip()
                            + "\n"
                        )
                    else:
                        temp_cache = line + "\n"

                    if title["end_text"] == "empty":
                        value.append(temp_cache)
                        is_get_value = False
                        temp_cache = ""
                        continue
                    else:
                        is_get_value = True
                        continue
    return value, paragraph_title



In [3]:
# from pdf2image import convert_from_path
# images = convert_from_path('/home/duckq1u/Documents/obsidian_aio/Notebook/Môn học trên trường/OCR thầy minh/data_test/0.pdf')
# image_output, paragraph = rounding_box(images[0], is_layout=False)
# print(paragraph)

# value, paragraph_title = get_text_from_paragraph(
#     paragraph.split("\n"), json_test
# )
# print("Paragraph Title:", paragraph_title)
# print("Extracted Values:")
# for i, val in enumerate(value):
#     print(val.strip())

## Show up an effciency model

In [4]:
# %pip install sacrebleu bert-score

In [5]:
import json

# NOTE: Change this to your folder path
folder_path = '/home/duckq1u/Documents/obsidian_aio/Notebook/Môn học trên trường/OCR thầy minh/data_test'

# Open and load the file
with open(folder_path+'/infor.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    
print(type(data))

<class 'dict'>


In [6]:

import os
from PIL import Image
from pdf2image import convert_from_path
from sacrebleu import corpus_bleu
from bert_score import score

def get_text_from_json(data, item):
    result = ""
    for index, value in enumerate(data[f'{item}']): 
        key = list(data[f'{item}'].keys())[index]
        value = data[f'{item}'][key]
        result = f"{result}\n{key} {value}"
    return result

def evaluate_similarity(reference: str, candidate: str, lang="en"):
    # BLEU Score 
    bleu = corpus_bleu([candidate], [[reference]])
    print(f"[BLEU] Similarity Score: {bleu.score:.2f}")

    # BERTScore 
    P, R, F1 = score([candidate], [reference], lang=lang)
    print(f"[BERTScore] Precision: {P[0]:.4f}, Recall: {R[0]:.4f}, F1: {F1[0]:.4f}")

# Loop through all files in the folder
for index, filename in enumerate(os.listdir(folder_path)):
    if filename.endswith('.pdf'): # Ignore json files
        file_path = os.path.join(folder_path, filename) # Construct full file path
        print(file_path)
        # Convert PDF to images
        images = convert_from_path(file_path)
        
        image = images[0]  # Assuming you want to process the first page
        image_output, paragraphs = rounding_box(image, is_layout=False)
        predict_paragraphs = ""
        for line in get_text_from_paragraph(paragraph=paragraphs.split("\n"), text_value=json_test)[0]:
            predict_paragraphs += line
        print(f"Predict paragraphs: {predict_paragraphs}")
        result_paragraphs = get_text_from_json(data, int(filename.split('.')[0]))
        
        evaluate_similarity(reference=result_paragraphs, candidate=predict_paragraphs, lang="vi")
        print('*'*50)

/home/duckq1u/Documents/obsidian_aio/Notebook/Môn học trên trường/OCR thầy minh/data_test/3.pdf


Recognizing Text: 100%|██████████| 35/35 [00:02<00:00, 15.65it/s]


[OCRResult(text_lines=[TextLine(polygon=[[1337.0, 82.0], [1503.0, 82.0], [1503.0, 112.0], [1337.0, 112.0]], confidence=0.9826072454452515, text='Mẫu số 05d', chars=[TextChar(polygon=[[1343.0, 82.0], [1374.0, 82.0], [1374.0, 111.0], [1343.0, 111.0]], confidence=0.9756737947463989, text='M', bbox_valid=True, bbox=[1343.0, 82.0, 1374.0, 111.0]), TextChar(polygon=[[1372.0, 82.0], [1388.0, 82.0], [1388.0, 111.0], [1372.0, 111.0]], confidence=0.9988732933998108, text='ẫ', bbox_valid=True, bbox=[1372.0, 82.0, 1388.0, 111.0]), TextChar(polygon=[[1387.0, 82.0], [1404.0, 82.0], [1404.0, 111.0], [1387.0, 111.0]], confidence=0.9915755391120911, text='u', bbox_valid=True, bbox=[1387.0, 82.0, 1404.0, 111.0]), TextChar(polygon=[[1403.0, 82.0], [1411.0, 82.0], [1411.0, 111.0], [1403.0, 111.0]], confidence=0.975335955619812, text=' ', bbox_valid=True, bbox=[1403.0, 82.0, 1411.0, 111.0]), TextChar(polygon=[[1411.0, 82.0], [1422.0, 82.0], [1422.0, 111.0], [1411.0, 111.0]], confidence=0.990254819393158, t

Recognizing Text: 100%|██████████| 37/37 [00:02<00:00, 14.68it/s]


[OCRResult(text_lines=[TextLine(polygon=[[1337.0, 82.0], [1503.0, 82.0], [1503.0, 112.0], [1337.0, 112.0]], confidence=0.9826298654079437, text='Mẫu số 05d', chars=[TextChar(polygon=[[1343.0, 82.0], [1374.0, 82.0], [1374.0, 111.0], [1343.0, 111.0]], confidence=0.9755208492279053, text='M', bbox_valid=True, bbox=[1343.0, 82.0, 1374.0, 111.0]), TextChar(polygon=[[1372.0, 82.0], [1388.0, 82.0], [1388.0, 111.0], [1372.0, 111.0]], confidence=0.9988547563552856, text='ẫ', bbox_valid=True, bbox=[1372.0, 82.0, 1388.0, 111.0]), TextChar(polygon=[[1387.0, 82.0], [1404.0, 82.0], [1404.0, 111.0], [1387.0, 111.0]], confidence=0.9910508394241333, text='u', bbox_valid=True, bbox=[1387.0, 82.0, 1404.0, 111.0]), TextChar(polygon=[[1403.0, 82.0], [1411.0, 82.0], [1411.0, 111.0], [1403.0, 111.0]], confidence=0.9735802412033081, text=' ', bbox_valid=True, bbox=[1403.0, 82.0, 1411.0, 111.0]), TextChar(polygon=[[1411.0, 82.0], [1422.0, 82.0], [1422.0, 111.0], [1411.0, 111.0]], confidence=0.9902250170707703,

...

[OCRResult(text_lines=[TextLine(polygon=[[1337.0, 82.0], [1503.0, 82.0], [1503.0, 112.0], [1337.0, 112.0]], confidence=0.9825380325317383, text='Mẫu số 05d', chars=[TextChar(polygon=[[1343.0, 82.0], [1374.0, 82.0], [1374.0, 111.0], [1343.0, 111.0]], confidence=0.9748295545578003, text='M', bbox_valid=True, bbox=[1343.0, 82.0, 1374.0, 111.0]), TextChar(polygon=[[1372.0, 82.0], [1388.0, 82.0], [1388.0, 111.0], [1372.0, 111.0]], confidence=0.9988788962364197, text='ẫ', bbox_valid=True, bbox=[1372.0, 82.0, 1388.0, 111.0]), TextChar(polygon=[[1387.0, 82.0], [1404.0, 82.0], [1404.0, 111.0], [1387.0, 111.0]], confidence=0.9910282492637634, text='u', bbox_valid=True, bbox=[1387.0, 82.0, 1404.0, 111.0]), TextChar(polygon=[[1403.0, 82.0], [1411.0, 82.0], [1411.0, 111.0], [1403.0, 111.0]], confidence=0.97531658411026, text=' ', bbox_valid=True, bbox=[1403.0, 82.0, 1411.0, 111.0]), TextChar(polygon=[[1411.0, 82.0], [1422.0, 82.0], [1422.0, 111.0], [1411.0, 111.0]], confidence=0.9903259873390198, t

Recognizing Text: 100%|██████████| 36/36 [00:02<00:00, 14.76it/s]


[OCRResult(text_lines=[TextLine(polygon=[[1337.0, 82.0], [1503.0, 82.0], [1503.0, 112.0], [1337.0, 112.0]], confidence=0.9827394664287568, text='Mẫu số 05d', chars=[TextChar(polygon=[[1344.0, 82.0], [1374.0, 82.0], [1374.0, 111.0], [1344.0, 111.0]], confidence=0.9748110771179199, text='M', bbox_valid=True, bbox=[1344.0, 82.0, 1374.0, 111.0]), TextChar(polygon=[[1372.0, 82.0], [1388.0, 82.0], [1388.0, 111.0], [1372.0, 111.0]], confidence=0.9988859295845032, text='ẫ', bbox_valid=True, bbox=[1372.0, 82.0, 1388.0, 111.0]), TextChar(polygon=[[1387.0, 82.0], [1404.0, 82.0], [1404.0, 111.0], [1387.0, 111.0]], confidence=0.9915926456451416, text='u', bbox_valid=True, bbox=[1387.0, 82.0, 1404.0, 111.0]), TextChar(polygon=[[1403.0, 82.0], [1411.0, 82.0], [1411.0, 111.0], [1403.0, 111.0]], confidence=0.9751777648925781, text=' ', bbox_valid=True, bbox=[1403.0, 82.0, 1411.0, 111.0]), TextChar(polygon=[[1411.0, 82.0], [1422.0, 82.0], [1422.0, 111.0], [1411.0, 111.0]], confidence=0.9902702569961548,

Recognizing Text: 100%|██████████| 36/36 [00:03<00:00, 11.90it/s]


[OCRResult(text_lines=[TextLine(polygon=[[1337.0, 82.0], [1503.0, 82.0], [1503.0, 112.0], [1337.0, 112.0]], confidence=0.9827103972434997, text='Mẫu số 05d', chars=[TextChar(polygon=[[1343.0, 82.0], [1374.0, 82.0], [1374.0, 111.0], [1343.0, 111.0]], confidence=0.9746203422546387, text='M', bbox_valid=True, bbox=[1343.0, 82.0, 1374.0, 111.0]), TextChar(polygon=[[1372.0, 82.0], [1388.0, 82.0], [1388.0, 111.0], [1372.0, 111.0]], confidence=0.9988705515861511, text='ẫ', bbox_valid=True, bbox=[1372.0, 82.0, 1388.0, 111.0]), TextChar(polygon=[[1387.0, 82.0], [1404.0, 82.0], [1404.0, 111.0], [1387.0, 111.0]], confidence=0.9909895658493042, text='u', bbox_valid=True, bbox=[1387.0, 82.0, 1404.0, 111.0]), TextChar(polygon=[[1403.0, 82.0], [1411.0, 82.0], [1411.0, 111.0], [1403.0, 111.0]], confidence=0.975219190120697, text=' ', bbox_valid=True, bbox=[1403.0, 82.0, 1411.0, 111.0]), TextChar(polygon=[[1411.0, 82.0], [1422.0, 82.0], [1422.0, 111.0], [1411.0, 111.0]], confidence=0.9902263879776001, 